# Hi-res 1601 — non-image feature zoo (modal / indicators / FRF) · GPU

The scientifically-sound complement to the CFDAC-image notebooks: the feature families that transferred **best** in the 128 baseline (modal, indicators) plus raw FRF, with **proper MLP / RandomForest / XGBoost / 1-D CNN / transformer**. Features are standardised on the synth-train fold and applied zero-shot to experiment; NN models train to convergence with checkpoint/resume; trees fit once. Set a GPU runtime; add a `GH_TOKEN` secret for autosave to `colab-hires-tabular`.

## 1 · Bootstrap

In [1]:
import os, sys, subprocess
GH_USER='grcarmenaty'; WORK='/content'; os.chdir(WORK)
def _tok():
    try:
        from google.colab import userdata; return userdata.get('GH_TOKEN')
    except Exception: return os.environ.get('GH_TOKEN')
def clone(repo, branch, dst):
    if os.path.isdir(dst): print('exists', dst); return
    t=_tok(); auth=f'{t}@' if t else ''
    url=f'https://{auth}github.com/{GH_USER}/{repo}.git'
    assert subprocess.run(['git','clone','--depth','1','-b',branch,url,dst]).returncode==0, \
        f'clone failed {repo}@{branch} (private? add a GH_TOKEN Colab secret)'
clone('phd_lanl','main','/content/PhD_LANL')
clone('pymodal','master','/content/pymodal')   # sibling dir the scripts expect
for p in ('/content/PhD_LANL','/content/pymodal'):
    if p not in sys.path: sys.path.insert(0,p)
os.chdir('/content/PhD_LANL')
subprocess.run([sys.executable,'-m','pip','-q','install','timm','h5py','scikit-learn','pint','pyFRF','audiomentations'])
import torch
print('torch',torch.__version__,'| cuda',torch.cuda.is_available(),'|',
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU - set a GPU runtime!')

exists /content/PhD_LANL
exists /content/pymodal
torch 2.11.0+cu128 | cuda True | Tesla T4


## 2 · Regenerate 1601-bin features

In [2]:
import subprocess, sys, os, glob, json, h5py, numpy as np
from pathlib import Path
REPO=Path(os.getcwd())
def run(cmd): print('>>',' '.join(cmd)); assert subprocess.run(cmd).returncode==0, cmd
if not (REPO/'dataset'/'features_hires.h5').exists():
    run([sys.executable,'ml_pipeline/generate_dataset.py','--out','dataset_hires','--n-t','4096','--fs','256'])
    run([sys.executable,'ml_pipeline/build_hires_synth_features.py'])
if not (REPO/'experimental_frfs.h5').exists():
    with open('experimental_frfs.h5','wb') as o:
        for p in sorted(glob.glob('experimental_frfs_chunks/experimental_frfs.h5.part_*')):
            o.write(open(p,'rb').read())
if not (REPO/'dataset'/'experimental_features.h5').exists():
    from ml_pipeline.evaluate import primary_op
    with h5py.File('experimental_frfs.h5','r') as f: names=json.loads(f.attrs['case_names_json'])
    n=len(names); tc=np.zeros(n,np.int8); st=np.full(n,-1,np.int8); en=np.full(n,-1,np.int8); sv=np.zeros(n,np.float32)
    for i,nm in enumerate(names):
        op=primary_op(nm); tc[i]=op['type_code']; st[i]=op['storey']; en[i]=op['end']; sv[i]=op['severity']
    dt=h5py.string_dtype('utf-8')
    with h5py.File('dataset/experimental_features.h5','w') as o:
        o.create_dataset('names',data=np.array(names,dtype=object),dtype=dt)
        o.create_dataset('type_code',data=tc); o.create_dataset('storey',data=st)
        o.create_dataset('end',data=en); o.create_dataset('severity',data=sv)
if not (REPO/'dataset'/'experimental_features_hires.h5').exists():
    run([sys.executable,'ml_pipeline/build_hires_exp_features.py'])
print('features ready')

features ready


## 3 · Config + precompute feature caches (once, on Drive)

In [3]:
import torch, numpy as np, h5py
from pathlib import Path
from ml_pipeline import hires_tab as T
from ml_pipeline.tasks import build_targets
from ml_pipeline.train import make_split
DEV = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ===================== CONFIG (edit me) =====================
MODELS = ['mlp','rf','xgb','cnn1d','transformer1d']
TASKS  = ['binary','col_location','mass_location','severity','type',
          'is_bolt','is_crack','is_mass','is_hole','is_pristine']
SUBSAMPLE = 4000          # synth samples per task (raise toward 10000 for more data)
BATCH     = 256           # tabular/seq NN batch (tiny models -> large batch fine)
AUTOSAVE_GITHUB   = True
FAMILY            = 'tabular'
GH_RESULTS_BRANCH = 'colab-hires-tabular'
# feature/model compatibility lives in T.TAB_MODEL_FEATURES:
#   mlp: modal,indicators,frf_mag,frf_realimag,timeseries | rf,xgb: modal,indicators
#   cnn1d,transformer1d: frf_mag,frf_realimag,timeseries
# NB timeseries is reconstructed from the FRF (IFFT*chirp) identically for synth &
# exp — the experimental set has no measured timeseries. To run ONE cell:
#   CELLS = [('is_hole','cnn1d','timeseries')]
CELLS = T.tab_cells(MODELS, TASKS)
print(len(CELLS),'cells queued across', MODELS)
# ===========================================================

try:
    from google.colab import drive; drive.mount('/content/drive')
    OUT = Path('/content/drive/MyDrive/hires_cfdac/tabular')
except Exception:
    OUT = Path('results_hires_zoo_tabular')
OUT.mkdir(parents=True, exist_ok=True); (OUT/'cache').mkdir(exist_ok=True)

SYN = 'dataset/features_hires.h5'; EXP = 'dataset/experimental_features_hires.h5'
with h5py.File(SYN,'r') as f:
    syn_tasks = build_targets(f['type_code'][:].astype('int64'),f['storey'][:].astype('int64'),
                              f['end'][:].astype('int64'),f['severity'][:].astype('float32'))
with h5py.File(EXP,'r') as f:
    exp_tasks = build_targets(f['type_code'][:].astype('int64'),f['storey'][:].astype('int64'),
                              f['end'][:].astype('int64'),f['severity'][:].astype('float32'))
    exp_names = [str(s) for s in f['names'][:]]

# Precompute each feature ONCE for ALL samples (cached on Drive), so indicators
# (a 1601 CFDAC per sample) are never recomputed per task.
feats_used = sorted({f for (_,_,f) in CELLS})
CACHE = {}
for ft in feats_used:
    print('building cache:', ft, '(indicators = slow: 1601 CFDAC/sample)')
    Xs = T.build_feature_cache(SYN, ft, OUT/'cache'/f'{ft}_syn.npy')
    Xe = T.build_feature_cache(EXP, ft, OUT/'cache'/f'{ft}_exp.npy')
    CACHE[ft] = (Xs, Xe)
print('caches:', {k:(tuple(v[0].shape),tuple(v[1].shape)) for k,v in CACHE.items()})
print('device', DEV, '| amp', T._amp_dtype(DEV))

150 cells queued across ['mlp', 'rf', 'xgb', 'cnn1d', 'transformer1d']
Mounted at /content/drive
building cache: frf_mag (indicators = slow: 1601 CFDAC/sample)
building cache: frf_realimag (indicators = slow: 1601 CFDAC/sample)
building cache: indicators (indicators = slow: 1601 CFDAC/sample)
building cache: modal (indicators = slow: 1601 CFDAC/sample)
building cache: timeseries (indicators = slow: 1601 CFDAC/sample)
cached timeseries (10000, 9, 4096) -> /content/drive/MyDrive/hires_cfdac/tabular/cache/timeseries_syn.npy
cached timeseries (2638, 9, 4096) -> /content/drive/MyDrive/hires_cfdac/tabular/cache/timeseries_exp.npy
caches: {'frf_mag': ((10000, 9, 1601), (2638, 9, 1601)), 'frf_realimag': ((10000, 18, 1601), (2638, 18, 1601)), 'indicators': ((10000, 22), (2638, 22)), 'modal': ((10000, 81), (2638, 81)), 'timeseries': ((10000, 9, 4096), (2638, 9, 4096))}
device cuda | amp torch.bfloat16


## 4 · Run the grid (skip-if-exists; NN resume from checkpoint)

In [4]:
import torch, os, shutil, subprocess
def _tok():
    try:
        from google.colab import userdata; return userdata.get('GH_TOKEN')
    except Exception: return os.environ.get('GH_TOKEN')
GH_TOKEN = _tok()
if AUTOSAVE_GITHUB and not GH_TOKEN:
    print('AUTOSAVE on but no GH_TOKEN -> Drive/zip only')

def git_autosave(msg):
    if not (AUTOSAVE_GITHUB and GH_TOKEN): return
    repo='/content/PhD_LANL'; dst=os.path.join(repo,'results_hires_zoo',FAMILY)
    os.makedirs(os.path.join(dst,'per_case'), exist_ok=True)
    for fn in (os.listdir(os.path.join(OUT,'per_case')) if os.path.isdir(os.path.join(OUT,'per_case')) else []):
        if fn.endswith('.json'): shutil.copy(os.path.join(OUT,'per_case',fn), os.path.join(dst,'per_case',fn))
    if os.path.exists(os.path.join(OUT,'synth_test_tab.json')):
        shutil.copy(os.path.join(OUT,'synth_test_tab.json'), os.path.join(dst,'synth_test_tab.json'))
    cwd=os.getcwd(); os.chdir(repo)
    subprocess.run(['git','config','user.email','colab@gpu.run'])
    subprocess.run(['git','config','user.name','colab-gpu'])
    subprocess.run(['git','add','-f',f'results_hires_zoo/{FAMILY}/per_case',f'results_hires_zoo/{FAMILY}/synth_test_tab.json'])
    if subprocess.run(['git','diff','--cached','--quiet']).returncode!=0:
        subprocess.run(['git','commit','-q','-m',msg])
        url=f'https://{GH_TOKEN}@github.com/grcarmenaty/phd_lanl.git'
        r=subprocess.run(['git','push','--force',url,f'HEAD:{GH_RESULTS_BRANCH}'],capture_output=True,text=True)
        print('  autosave:', f'pushed -> {GH_RESULTS_BRANCH}' if r.returncode==0 else 'FAILED '+r.stderr[-160:])
    os.chdir(cwd)

for (task, model, feature) in CELLS:
    try:
        T.run_tab_cell(task, model, feature, out_dir=OUT, dev=DEV, syn_tasks=syn_tasks,
                       exp_tasks=exp_tasks, Xsyn=CACHE[feature][0], Xexp=CACHE[feature][1],
                       exp_names=exp_names, make_split=make_split, subsample=SUBSAMPLE, batch=BATCH)
        git_autosave(f'colab autosave [tabular]: {task}/{model}/{feature}')
    except Exception as e:
        print('CELL FAILED', task, model, feature, '::', repr(e)[:200])
        if torch.cuda.is_available(): torch.cuda.empty_cache()
print('\nqueue done')

    binary_mlp_modal_hires1601 ep10 val=+0.8863 best=+0.9174 since=3 (21s)
    binary_mlp_modal_hires1601 ep20 val=+0.9261 best=+0.9395 since=1 (21s)
    binary_mlp_modal_hires1601 ep30 val=+0.9350 best=+0.9532 since=3 (22s)
    binary_mlp_modal_hires1601 ep40 val=+0.9486 best=+0.9556 since=2 (23s)
    binary_mlp_modal_hires1601 ep50 val=+0.9509 best=+0.9556 since=12 (23s)
    binary_mlp_modal_hires1601 ep53 val=+0.9509 best=+0.9556 since=15 (23s)
    binary_mlp_modal_hires1601: converged
  DONE binary_mlp_modal_hires1601: synth mF1=0.937
  autosave: pushed -> colab-hires-tabular
    binary_mlp_indicators_hires1601 ep10 val=+0.7075 best=+0.7351 since=2 (1s)
    binary_mlp_indicators_hires1601 ep20 val=+0.8239 best=+0.8239 since=0 (1s)
    binary_mlp_indicators_hires1601 ep30 val=+0.8368 best=+0.8496 since=3 (1s)
    binary_mlp_indicators_hires1601 ep40 val=+0.8224 best=+0.8584 since=1 (2s)
    binary_mlp_indicators_hires1601 ep50 val=+0.8744 best=+0.8843 since=2 (2s)
    binary_mlp_ind

/content/PhD_LANL/ml_pipeline/hires_tab.py:203: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.enc = nn.TransformerEncoder(enc, depth); self.norm = nn.LayerNorm(dim)


    binary_transformer1d_frf_mag_hires1601 ep10 val=+0.1667 best=+0.4444 since=9 (15s)
    binary_transformer1d_frf_mag_hires1601 ep20 val=+0.5353 best=+0.6101 since=4 (29s)
    binary_transformer1d_frf_mag_hires1601 ep30 val=+0.6303 best=+0.6343 since=1 (44s)
    binary_transformer1d_frf_mag_hires1601 ep40 val=+0.5625 best=+0.6931 since=2 (58s)
    binary_transformer1d_frf_mag_hires1601 ep50 val=+0.5877 best=+0.6931 since=12 (72s)
    binary_transformer1d_frf_mag_hires1601 ep53 val=+0.6806 best=+0.6931 since=15 (76s)
    binary_transformer1d_frf_mag_hires1601: converged
  DONE binary_transformer1d_frf_mag_hires1601: synth mF1=0.603
  autosave: pushed -> colab-hires-tabular


/content/PhD_LANL/ml_pipeline/hires_tab.py:203: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.enc = nn.TransformerEncoder(enc, depth); self.norm = nn.LayerNorm(dim)


    binary_transformer1d_frf_realimag_hires1601 ep10 val=+0.7134 best=+0.7134 since=0 (15s)
    binary_transformer1d_frf_realimag_hires1601 ep20 val=+0.6813 best=+0.7282 since=8 (30s)
    binary_transformer1d_frf_realimag_hires1601 ep30 val=+0.7347 best=+0.7571 since=1 (44s)
    binary_transformer1d_frf_realimag_hires1601 ep40 val=+0.7676 best=+0.7676 since=0 (59s)
    binary_transformer1d_frf_realimag_hires1601 ep50 val=+0.7596 best=+0.7793 since=8 (74s)
    binary_transformer1d_frf_realimag_hires1601 ep60 val=+0.7927 best=+0.7927 since=0 (88s)
    binary_transformer1d_frf_realimag_hires1601 ep70 val=+0.7935 best=+0.8130 since=1 (103s)
    binary_transformer1d_frf_realimag_hires1601 ep80 val=+0.8689 best=+0.8689 since=0 (118s)
    binary_transformer1d_frf_realimag_hires1601 ep90 val=+0.8766 best=+0.8873 since=5 (132s)
    binary_transformer1d_frf_realimag_hires1601 ep100 val=+0.8878 best=+0.8980 since=4 (147s)
    binary_transformer1d_frf_realimag_hires1601 ep110 val=+0.8954 best=+0.9

...

    is_pristine_transformer1d_frf_mag_hires1601 ep10 val=+0.5950 best=+0.6047 since=2 (14s)
    is_pristine_transformer1d_frf_mag_hires1601 ep20 val=+0.5474 best=+0.6061 since=7 (28s)
    is_pristine_transformer1d_frf_mag_hires1601 ep28 val=+0.4973 best=+0.6061 since=15 (39s)
    is_pristine_transformer1d_frf_mag_hires1601: converged
  DONE is_pristine_transformer1d_frf_mag_hires1601: synth mF1=0.665
  autosave: pushed -> colab-hires-tabular


/content/PhD_LANL/ml_pipeline/hires_tab.py:203: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.enc = nn.TransformerEncoder(enc, depth); self.norm = nn.LayerNorm(dim)


    is_pristine_transformer1d_frf_realimag_hires1601 ep10 val=+0.5987 best=+0.6097 since=1 (15s)
    is_pristine_transformer1d_frf_realimag_hires1601 ep20 val=+0.5652 best=+0.6352 since=4 (30s)
    is_pristine_transformer1d_frf_realimag_hires1601 ep30 val=+0.6756 best=+0.6756 since=0 (45s)
    is_pristine_transformer1d_frf_realimag_hires1601 ep40 val=+0.7120 best=+0.7389 since=1 (60s)
    is_pristine_transformer1d_frf_realimag_hires1601 ep50 val=+0.7078 best=+0.7601 since=6 (74s)
    is_pristine_transformer1d_frf_realimag_hires1601 ep59 val=+0.7326 best=+0.7601 since=15 (87s)
    is_pristine_transformer1d_frf_realimag_hires1601: converged
  DONE is_pristine_transformer1d_frf_realimag_hires1601: synth mF1=0.755
  autosave: pushed -> colab-hires-tabular


/content/PhD_LANL/ml_pipeline/hires_tab.py:203: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.enc = nn.TransformerEncoder(enc, depth); self.norm = nn.LayerNorm(dim)


    is_pristine_transformer1d_timeseries_hires1601 ep10 val=+0.4894 best=+0.6139 since=5 (32s)
    is_pristine_transformer1d_timeseries_hires1601 ep20 val=+0.6170 best=+0.6776 since=1 (62s)
    is_pristine_transformer1d_timeseries_hires1601 ep30 val=+0.7279 best=+0.7483 since=3 (94s)
    is_pristine_transformer1d_timeseries_hires1601 ep40 val=+0.8384 best=+0.8466 since=4 (125s)
    is_pristine_transformer1d_timeseries_hires1601 ep50 val=+0.8748 best=+0.8748 since=0 (156s)
    is_pristine_transformer1d_timeseries_hires1601 ep60 val=+0.8835 best=+0.8867 since=4 (187s)
    is_pristine_transformer1d_timeseries_hires1601 ep70 val=+0.8883 best=+0.8964 since=6 (218s)
    is_pristine_transformer1d_timeseries_hires1601 ep79 val=+0.8916 best=+0.8964 since=15 (246s)
    is_pristine_transformer1d_timeseries_hires1601: converged
  DONE is_pristine_transformer1d_timeseries_hires1601: synth mF1=0.877
  autosave: pushed -> colab-hires-tabular

queue done


## 5 · Honest summary + zip download

In [5]:
import json, numpy as np
from pathlib import Path
from collections import Counter
from sklearn.metrics import balanced_accuracy_score, f1_score, accuracy_score
print(f"{'cell':<46}{'kind':>5}{'synth':>8}{'expMF1/R2':>11}{'expBal':>8}{'collapse':>9}")
print('-'*86)
for p in sorted((OUT/'per_case').glob('*_hires1601.json')):
    d=json.loads(p.read_text()); m=d['meta']; r=d['rows']
    yt=np.array([x['y_true'] for x in r]); yp=np.array([x['y_pred'] for x in r])
    name=f"{m['task']}/{m['model']}/{m['feature']}"
    if m['kind']=='cls':
        n=m['n_out']; bal=balanced_accuracy_score(yt,yp)
        mf1=f1_score(yt,yp,labels=list(range(n)),average='macro',zero_division=0)
        coll=(len(set(yp.tolist()))<=1) or (bal<=1/n+0.02)
        print(f"{name:<46}{'cls':>5}{(m.get('synth_test_macro_f1') or 0):>8.3f}{mf1:>11.3f}{bal:>8.3f}{str(coll):>9}")
    else:
        yt=yt.astype(float); yp=yp.astype(float); ss=np.sum((yt-yp)**2); st=np.sum((yt-yt.mean())**2)
        r2=1-ss/st if st>0 else 0
        print(f"{name:<46}{'reg':>5}{(m.get('synth_test_metric') or 0):>8.3f}{r2:>11.3f}{'-':>8}{'-':>9}")
import shutil
shutil.make_archive('/content/results_tabular','zip',str(OUT))
try:
    from google.colab import files; files.download('/content/results_tabular.zip')
except Exception as e: print('zip at /content/results_tabular.zip', e)

cell                                           kind   synth  expMF1/R2  expBal collapse
--------------------------------------------------------------------------------------
binary/cnn1d/frf_mag                            cls   0.739      0.452   0.500     True
binary/cnn1d/frf_realimag                       cls   0.725      0.452   0.500     True
binary/cnn1d/timeseries                         cls   0.772      0.452   0.500     True
binary/mlp/frf_mag                              cls   0.963      0.452   0.500     True
binary/mlp/frf_realimag                         cls   0.935      0.562   0.562    False
binary/mlp/indicators                           cls   0.831      0.452   0.500     True
binary/mlp/modal                                cls   0.937      0.452   0.500     True
binary/mlp/timeseries                           cls   0.951      0.490   0.516     True
binary/rf/indicators                            cls   0.791      0.452   0.500     True
binary/rf/modal                  

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 6 · (Optional) push results to the repo

In [6]:
# Optional: push the JSON results to the repo (needs a GH_TOKEN secret with write).
import os, subprocess, shutil
tok=None
try:
    from google.colab import userdata; tok=userdata.get('GH_TOKEN')
except Exception: tok=os.environ.get('GH_TOKEN')
if not tok:
    print('No GH_TOKEN - hand the downloaded zip to the agent instead.')
else:
    dst='/content/PhD_LANL/results_hires_zoo'; os.makedirs(dst, exist_ok=True)
    if str(OUT)!=dst and (OUT/'per_case').exists():
        shutil.copytree(OUT, dst, dirs_exist_ok=True)
    os.chdir('/content/PhD_LANL')
    subprocess.run(['git','config','user.email','colab@gpu.run'])
    subprocess.run(['git','config','user.name','colab-gpu'])
    subprocess.run(['git','add','results_hires_zoo'])
    subprocess.run(['git','commit','-m','hires zoo (GPU): CFDAC cells synth+exp'])
    url=f'https://{tok}@github.com/grcarmenaty/phd_lanl.git'
    print(subprocess.run(['git','push',url,'HEAD:main'],capture_output=True,text=True).stderr[-400:])

error: RPC failed; HTTP 408 curl 22 The requested URL returned error: 408
send-pack: unexpected disconnect while reading sideband packet
fatal: the remote end hung up unexpectedly
Everything up-to-date

